# Detección de anomalías: Isolation Forest y One-Class SVM RBF

Este notebook compara ambos modelos sobre métricas de Prometheus o un CSV provisto por los docentes. Cambiar `INPUT_CSV` y, si corresponde, los nombres de columnas. El análisis es no supervisado y exploratorio: ajusta y puntúa el mismo dataset; si hay etiquetas, las métricas de evaluación son *in-sample*.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from pipeline import (AnalysisConfig, ISOLATION_FOREST, ONE_CLASS_SVM, analyze_dataset, load_csv, plot_comparison, save_artifacts)

## 1. Dataset y parámetros

El CSV de ejemplo proviene de Prometheus. Para un dataset docente, señalar su ruta y usar `FEATURE_COLUMNS` si incluye otras columnas numéricas que no sean métricas. `LABEL_COLUMN` se excluye siempre de las variables de entrada; `ANOMALY_LABEL` permite calcular precisión, recall y F1 si hay etiquetas.

In [ ]:
INPUT_CSV = Path('examples/prometheus_sample.csv')
TIMESTAMP_COLUMN = 'timestamp'  # None si el CSV no tiene fecha
FEATURE_COLUMNS = None  # p. ej. ['cpu_percent', 'memory_percent']
LABEL_COLUMN = None  # nombre de la columna de etiquetas, si existe
ANOMALY_LABEL = None  # p. ej. '1'; solo junto con LABEL_COLUMN
PLOT_FEATURE = 'cpu_percent'  # None para usar la primera métrica
config = AnalysisConfig(contamination=0.05, nu=0.05, gamma='scale', random_state=42)
frame = load_csv(INPUT_CSV)
print(f'{len(frame)} filas, {len(frame.columns)} columnas')
display(frame.head())

## 2. Ajustar ambos modelos

Ambos reciben las mismas métricas numéricas, con valores faltantes imputados por mediana y escala estándar. `contamination` fija el umbral de Isolation Forest; `nu` y `gamma` controlan One-Class SVM. Un score más alto en los resultados significa mayor anomalía; los scores entre modelos no tienen la misma escala.

In [ ]:
models = {
    ISOLATION_FOREST: IsolationForest(
        n_estimators=config.n_estimators, contamination=config.contamination,
        random_state=config.random_state, n_jobs=-1,
    ),
    ONE_CLASS_SVM: OneClassSVM(kernel='rbf', nu=config.nu, gamma=config.gamma),
}
result = analyze_dataset(
    frame, config, models=models, feature_columns=FEATURE_COLUMNS,
    timestamp_column=TIMESTAMP_COLUMN, label_column=LABEL_COLUMN,
    anomaly_label=ANOMALY_LABEL,
)
print('Métricas utilizadas:', result.feature_columns)
display(pd.DataFrame(result.summary['models']).T)
display(pd.DataFrame([result.summary['comparison']]))

## 3. Comparar filas y visualizar anomalías

La gráfica marca las anomalías detectadas por cada modelo sobre la misma métrica. Revisar también las filas marcadas y la intersección: los modelos pueden discrepar porque sus parámetros no representan exactamente el mismo porcentaje de anomalías.

In [ ]:
display(result.scored.loc[
    (result.scored['isolation_forest_anomaly'] == 1) |
    (result.scored['one_class_svm_rbf_anomaly'] == 1)
].head(20))
figure = plot_comparison(result, feature=PLOT_FEATURE, timestamp_column=TIMESTAMP_COLUMN)
display(figure)

In [ ]:
paths = save_artifacts(result, Path('output/notebook'), plot_feature=PLOT_FEATURE, timestamp_column=TIMESTAMP_COLUMN)
print(paths)